In [ ]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [ ]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [ ]:
run_sql_file('../sql/06_customer_features.sql')

In [ ]:
q("""
SELECT
    connection_type,
    COUNT(*) AS total_subscribers,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY connection_type;
""")

In [ ]:
q("""
SELECT
    plan_name,
    plan_type,
    price_inr,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY total_subscribers DESC
LIMIT 5;
""")

In [ ]:
q("""
SELECT
    circle_name,
    region,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY circle_name, region
ORDER BY total_subscribers DESC
LIMIT 5;
""")

In [ ]:
q("""
SELECT
    age_band,
    COUNT(*) AS subs_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS s_percentage
FROM customer_features
GROUP BY age_band
ORDER BY subs_count DESC;
""")

In [ ]:
q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subscriber_count DESC;
""")

In [ ]:
q("""
SELECT
    device_type,
    os,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY device_type,os
ORDER BY device_type,os""")

In [ ]:
q("""
SELECT
    tenure_years,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

In [ ]:
q("""SELECT
    AVG(stated_arpu) AS average_monthly_revenue
FROM customer_features;""")

In [ ]:
q("""SELECT
    AVG(actual_arpu_6m) AS average_actual_arpu,
    SUM(actual_arpu_6m) AS total_monthly_revenue
FROM customer_features;""")

In [ ]:
q("""select
ROUND(AVG(stated_arpu),2)as avg_st_arpu, 
ROUND(AVG(actual_arpu_6m),2)as act_arpu_6pm,
ROUND(AVG(stated_arpu - actual_arpu_6m),2)as gap,
ROUND(
        AVG(stated_arpu - actual_arpu_6m)
        * 100.0 / AVG(actual_arpu_6m), 2
    ) AS gap_percentage
FROM customer_features;
""")

In [ ]:
q("""select
    connection_type,
    count(*) as subs,
    AVG(stated_arpu)as avg_st_arpu,
    AVG(actual_arpu_6m)as avg_arpu_6m,
    AVG(stated_arpu - actual_arpu_6m)as gap
    
    FROM customer_features
    GROUP BY connection_type;""")

In [ ]:
q("""select
    plan_name,
    count(*) as total_subscribers,
    SUM(actual_arpu_6m) as sum_aprpu,
    Avg(actual_arpu_6m) as avg_aprpu
    FROM customer_features
    group by plan_name
    ORDER BY sum_aprpu DESC limit 5;
  






""")

In [ ]:
q("""SELECT
    circle_name,
    SUM(actual_arpu_6m) AS total_monthly_revenue
FROM customer_features
GROUP BY circle_name
ORDER BY total_monthly_revenue DESC
LIMIT 5;
""")

In [ ]:
q("""SELECT
    tenure_years,
    ROUND(AVG(actual_arpu_6m),2) AS avg_actual_arpu
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

In [ ]:
q("""SELECT
    txn_type,
    COUNT(*) AS total_payments,
    COUNT(DISTINCT customer_id) AS total_subscribers
FROM recharges
GROUP BY txn_type;
""")

In [ ]:
q("""select avg(amount_inr)as avg_amo
from recharges""")

In [ ]:
q("""SELECT
    txn_type,
    AVG(amount_inr) AS average_payment,
    MIN(amount_inr) AS minimum_payment,
    MAX(amount_inr) AS maximum_payment
FROM recharges
GROUP BY txn_type;
""")

In [ ]:
q("""SELECT
    connection_type,
    AVG(recharge_count) AS avg_recharge_count,
    AVG(recharge_total_6m) AS avg_recharge_amount
FROM customer_features
GROUP BY connection_type;
""")

In [ ]:
q("""SELECT
    channel,
    COUNT(*) AS total_payments,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM recharges), 2
    ) AS payment_share_percentage
FROM recharges
GROUP BY channel
ORDER BY total_payments DESC;
""")

In [ ]:
q("""SELECT
    recharge_count,
    COUNT(*) AS total_subscribers,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_percentage
FROM customer_features
WHERE recharge_count <= 8
GROUP BY recharge_count
ORDER BY recharge_count;
""")

In [ ]:
q("""SELECT
    month,
    SUM(total_mb) AS total_data_mb,
    ROUND(SUM(total_mb) / 1000000.0, 2) AS total_data_tb,
    COUNT(DISTINCT customer_id) AS total_subscribers
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

In [ ]:
q("""SELECT
    ROUND(AVG(data_mb), 2) AS avg_daily_data_mb
FROM usage_daily;
""")

In [ ]:
q("""SELECT
    AVG(mb_per_active_day) AS avg_per_active_day,
    AVG(mb_per_observed_day) AS avg_per_observed_day,
    AVG(mb_per_active_day) /
    AVG(mb_per_observed_day) AS usage_ratio
FROM customer_features;
""")

In [ ]:
q("""SELECT
    strftime('%w', usage_date) AS weekday,
    COUNT(*) AS total_records,
    ROUND(AVG(data_mb), 2) AS avg_data_mb
FROM usage_daily
GROUP BY weekday
ORDER BY weekday;
""")

In [ ]:
q("""
SELECT
    strftime('%m', usage_date) AS month,
    ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
    ROUND(AVG(sms_count), 2) AS avg_sms_count
FROM usage_daily
GROUP BY strftime('%m', usage_date)
ORDER BY month;
""")

In [ ]:
q("""
SELECT
    month,
    ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb,
    ROUND(AVG(usage_days), 2) AS avg_usage
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

In [ ]:
q("""
WITH quartile_data AS (
    SELECT
        customer_id,
        mb_per_active_day,
        NTILE(4) OVER (
            ORDER BY mb_per_active_day
        ) AS quartile
    FROM customer_features
    WHERE mb_per_active_day > 0
)
SELECT
    quartile,
    COUNT(*) AS subscriber_count,
    ROUND(MIN(mb_per_active_day), 2) AS min_data_mb,
    ROUND(MAX(mb_per_active_day), 2) AS max_data_mb
FROM quartile_data
GROUP BY quartile
ORDER BY quartile;
""")

In [ ]:
q("""
WITH quartile_data AS (
    SELECT
        customer_id,
        mb_per_active_day,
        is_churned,
        NTILE(4) OVER (
            ORDER BY mb_per_active_day
        ) AS quartile
    FROM customer_features
    WHERE mb_per_active_day > 0
)
SELECT
    quartile,
    COUNT(*) AS subscriber_count,
    ROUND(AVG(is_churned) * 100, 2) AS churn_percentage
FROM quartile_data
GROUP BY quartile
ORDER BY quartile;
""")


In [ ]:
q("""
SELECT
    connection_type,
    device_type,
    COUNT(*) AS subscriber_count,
    ROUND(AVG(total_voice_min / NULLIF(usage_days, 0)), 2) AS voice_per_active_day,
    ROUND(AVG(mb_per_active_day), 2) AS mb_per_active_day
FROM customer_features
WHERE usage_days > 0
GROUP BY connection_type, device_type
ORDER BY voice_per_active_day DESC;
""")

In [ ]:
q("""
SELECT
    m_score,
    COUNT(*) AS subscriber_count,
    ROUND(AVG(recharge_total_6m), 2) AS avg_spend,
    ROUND(AVG(mb_per_active_day), 2) AS avg_data_per_active_day,
    ROUND(AVG(is_churned) * 100, 2) AS churn_percentage
FROM customer_features
GROUP BY m_score
ORDER BY m_score;
""")

In [ ]:
q("""select round(avg(100*total_sms/nullif(usage_days,0)),2)as_sms_per_day,
sum(total_sms=0)as never_used,
count(*)as subs
from customer_features
where usage_days>0""")

In [ ]:
q("""select * from usage_daily """)

In [ ]:
q("""
SELECT
    is_roaming,
    COUNT(*) AS total_days,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS day_share_percent,
    COUNT(DISTINCT usage_id	) AS total_subscribers
FROM usage_daily
GROUP BY is_roaming;
""")

In [ ]:
q("""
SELECT
    connection_type,
    AVG(roaming_days) AS avg_roaming_days,
    COUNT(CASE WHEN roaming_days > 0 THEN 1 END) AS ever_roamed,
    COUNT(*) AS total_subscribers
FROM customer_features
GROUP BY connection_type;
""")

In [ ]:
q("""
SELECT
    CASE
        WHEN roaming_days = 0 THEN 'No roaming'
        WHEN roaming_days <= 5 THEN 'Low roaming'
        WHEN roaming_days <= 15 THEN 'Medium roaming'
        ELSE 'High roaming'
    END AS roaming_group,

    COUNT(*) AS total_subscribers,
    ROUND(AVG(is_Churned), 2) AS avg_churn

FROM customer_features
GROUP BY roaming_group
ORDER BY avg_churn;
""")

In [ ]:
q("""
 select  channel, count(*) as  tickets,
 round(100.0*sum(is_resolved)/count(*),2) as perc
 from complaints
 group by channel
 order by tickets  desc;
 """)

In [ ]:
# G3 · How many tickets are still open?
# STEPS
# 1. Group by is_resolved.
# # 2. Count and share.

q("""
select  is_resolved, count(*) as tickets,
round(100.0*count(*)/sum(count(*)) over (),2) as perc
from complaints
group by is_resolved
order by tickets  desc;
""")

In [ ]:
# 2 G5 · How many days does a resolved ticket take to close?
# STEPS
# 1. Filter to resolved tickets only.
# 2. Average, minimum and maximum resolution_days.
# 3. Say explicitly what population that is measured on.

q("""
 select count(*) as resolved_tickets,
 round(avg(resolution_days),2) as avg_resolution_days,
 min(resolution_days) as min_resolution_days,
 max(resolution_days) as max_resolution_days
 from complaints
 where is_resolved = 1;
 """)

In [ ]:
# How many subscribers have raised 0, 1, 2 or more complaints?
# STEPS
# 1. Group customer_features by complaint_count.
# 2. Count subscribers at each level.
# 3. Read the whole distribution rather than the average.

q("""
SELECT
    complaint_count,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY complaint_count
ORDER BY complaint_count;
""")

In [ ]:
# · Which five circles raise the most complaints per subscriber?
# STEPS
# 1. Group by circle_name.
# 2. Divide total tickets by subscribers so circles of different sizes compare.
# 3. Sort and take five

q("""
SELECT
   circle_name,
    COUNT(*) AS total_complaints,
    COUNT(DISTINCT customer_id) AS subscriber_count,
    COUNT(*) / COUNT(DISTINCT customer_id) AS complaints_per_subscriber
FROM customer_features
GROUP BY circle_name
ORDER BY subscriber_count DESC
LIMIT 5;
""")

In [ ]:
# H1 · How many of our current subscribers joined in each year?
# STEPS
# 1. Read customer_features.
# 2. Pull the year out of activation_date with strftime('%Y', …).
# 3. Group and count.

q("""
SELECT
    strftime('%Y', activation_date) AS join_year,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY join_year
ORDER BY join_year;
""")

In [ ]:
# H2 · What share of each joining year has left?
# STEPS
# 1. Same grouping.
# 2. Add the churn count and rate.
# 3. Read down the churn column.

q("""
SELECT
    strftime('%Y', activation_date) AS join_year,
    COUNT(*) AS subscriber_count,
    sum(is_churned) AS churn_count,
    round(100.0*avg(is_churned),2) AS churn_rate
FROM customer_features
GROUP BY join_year
ORDER BY join_year;
""")

In [ ]:
# H3 · How does churn change with each completed year on the network?
# STEPS
# 1. Group by tenure_years
# 2. Sort by tenure

q("""
SELECT
    tenure_years,
    COUNT(*) AS subscriber_count,
    
    round(100.0*avg(is_churned),2) AS churn_rate
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

In [ ]:
# H4 · How long had a leaver been with us when they left?
# STEPS
# 1. Filter to churned subscribers.
# 2. Average, minimum and maximum days_to_churn, which Step 2 built from the two dates.

q("""
  SELECT
    ROUND(AVG(days_to_churn), 1) AS avg_days,
    MIN(days_to_churn) AS min_days,
    MAX(days_to_churn) AS max_days,
    COUNT(*) as leaved_ones
    FROM customer_features
    WHERE is_churned = 1;
""")

In [ ]:
# · Do longer-standing subscribers pay more?
# STEPS
# 1. Group by tenure_years.
# # 2. Average both actual_arpu_6m and recharge_total_6m.

q("""
 SELECT
    tenure_years,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
    ROUND(AVG(recharge_total_6m), 2) AS avg_recharge_total
FROM customer_features
GROUP BY tenure_years;
""")

In [ ]:
# 1 · How many subscribers have we lost, and what is the churn rate?
# S T E P S
# 1. Count all subscribers.
# 2. Sum is_churned.
# 3. Average it for the rate.

q("""
SELECT  
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features;
""")

In [ ]:
# I2 · What is the churn rate for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Count, sum and average churn.

q("""
SELECT
    connection_type,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY connection_type;
""")

In [ ]:
# I3 · Which six plans have the highest churn rate?
# STEPS
# 1. Group by plan_name, carrying plan_type.
# 2. Count and average churn.
# 3. Sort worst first, take six.
q("""
SELECT
    plan_name,
    plan_type,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY plan_name, plan_type
ORDER BY churn_rate_percentage DESC
LIMIT 6;
""")

In [ ]:
# I4 · Which circles have the highest and the lowest churn rate?
# STEPS
# 1. Group by circle_name, carrying region.
#  2. Count and average churn.
# 3. Sort worst first and print all 22, because the spread is the finding.

q("""
SELECT
    circle_name,
    region,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn_rate_percentage DESC;
""")

In [ ]:
# I5 · Do smartphone users leave at a different rate from feature-phone users?
# STEPS
# 1. Group by device_type.
# 2. Count and average churn.
# 3. Read the two numbers.

q("""
SELECT
    device_type,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY device_type;
""")

In [ ]:
# · Does churn rise with each extra open ticket?
# S T E P S
# 1. Group by unresolved_count directly.
# 2. Count and average churn.
# 3. Read the gradient and then read the group sizes.

q("""
SELECT
    unresolved_count,
    COUNT(*) AS subs,
    ROUND(AVG(is_churned) * 100.0, 2) Churn
FROM customer_features
GROUP BY unresolved_count
ORDER BY unresolved_count;
""")

In [ ]:
# K1 · What is the churn rate in each recency quartile?
# S T E P S
# 1. Group by r_score.
# 2. Average churn.
# 3. Look at it before saying anything.

q("""

SELECT
    r_score,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS total_churned,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY r_score
ORDER BY r_score;
""")
 

In [ ]:
# K2 · What is the churn rate in each frequency quartile?
# STEPS
# 1. Group by f_score.
# 2. Average recharge_count to check the score means what it says.
# 3. Average churn.

q("""
SELECT
    f_score,
    COUNT(*) AS subs,
    ROUND(AVG(is_churned) * 100.0, 2) AS Churn
FROM customer_features
GROUP BY f_score
ORDER BY f_score;
""")

In [ ]:
# # L1 · What are the churn rate and the monthly revenue of each region?
# # STEPS
# # 1. Group by region.
# # Trainer-facing  ·  companion to the Airtel Trainer Manual Page 220
# # AIRTEL TELECOM ANALYTICS CLASSROOM DELIVERY RUNBOOK
# # 2. Count, sum revenue and average churn.
# # # 3. Sort by churn.

q("""
SELECT
    region,
    COUNT(*) AS total_subscribers,
    SUM(actual_arpu_6m) AS total_monthly_revenue,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_percentage
FROM customer_features
GROUP BY region
ORDER BY churn_rate_percentage DESC;
""")

In [ ]:
# L2 · Which four circles churn the most, and what do their subscribers pay?
# STEPS
# 1. Group by circle and region.
# 2. Average churn and ARPU together.
# 3. Sort worst first, take four.

q( """
SELECT
    circle_name,
    region,
    COUNT(*) AS subs,
    ROUND(AVG(is_churned) * 100.0, 2) AS churn,
    ROUND(AVG(actual_arpu_6m), 2) AS revenue
FROM customer_features
GROUP BY circle_name, region
ORDER BY  churn DESC
LIMIT 4;
""")

In [ ]:
# L7 · Do regions differ in revenue per subscriber and in data use?
# S T E P S
# 1. Group by region.
# 2. Average actual_arpu_6m and mb_per_active_day.
# 3. Sort by ARPU.

q("""
SELECT
    region,
    ROUND(AVG(actual_arpu_6m), 2) AS revenue,
    ROUND(AVG(mb_per_active_day), 2) AS avg_data_use
FROM customer_features
GROUP BY region
ORDER BY revenue DESC;
""")

In [ ]:
# · How many subscribers are in group A and in group B?
# S T E P S
# 1. Group customer_features by ab_group.
# 2. Count and share.

q("""
SELECT
    ab_group,
    COUNT(*) AS subscriber_count
    
FROM customer_features
GROUP BY ab_group;
""")

In [ ]:
# What is the churn rate in group A and in group B?
# STEPS
# 1. Group by ab_group.
# 2. Count, sum and average is_churned.
# 3. Read the two rates and then say nothing conclusive.

q("""
SELECT
    ab_group,
    COUNT(*) AS total_subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(SUM(is_churned) * 100.0 / COUNT(*), 2) AS churn_rate
FROM customer_features
GROUP BY ab_group;
""")

In [ ]:
# Does revenue per subscriber differ between the groups, on either ARPU column?
# 1. Group by ab_group.
# 2. Average both the stated and the measured ARPU column.
# 3. Compare the two rows, and then compare the two columns.

q("""
SELECT
    ab_group,
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu
FROM customer_features
GROUP BY ab_group;
""")